In [ ]:
import requests
import pandas as pd
from datetime import datetime
from calendar import monthrange
import os
import time

# Cấu hình
API_KEY = 'ab6016bf5295490383b6b0d24dc63ca3'
API_KEY_BACKUP = 'd3796dbb939c40a985f9573949115b4f'

# File outputs
WEATHER_FILE_TEST = 'hanoi_weather_history_test_data.csv'
AIR_QUALITY_FILE_TEST = 'hanoi_air_quality_history_test_data.csv'
FINAL_COMBINED_FILE = 'hanoi_weather_air_quality_final_test_data.csv'

# Tọa độ Hà Nội
HANOI_LAT = 21.0285
HANOI_LON = 105.8542

# Xác định khoảng thời gian
start_date = datetime(2025, 11, 1)
end_date = datetime(2025, 11, 7)

In [2]:
def get_weather_data(start_date, end_date, api_key):
    """
    Lấy dữ liệu thời tiết từ Weatherbit API (hourly history).
    
    Parameters:
    -----------
    start_date : str
        Ngày bắt đầu (định dạng: 'YYYY-MM-DD')
    end_date : str
        Ngày kết thúc (định dạng: 'YYYY-MM-DD')
    api_key : str
        API key của Weatherbit
        
    Returns:
    --------
    df : pandas DataFrame hoặc None
        DataFrame chứa dữ liệu thời tiết hoặc None nếu có lỗi
    """
    url = "https://api.weatherbit.io/v2.0/history/hourly"
    params = {
        'lat': HANOI_LAT,
        'lon': HANOI_LON,
        'start_date': start_date,
        'end_date': end_date,
        'tz': 'local',
        'key': api_key
    }

    try:
        print(f"    Đang gọi Weather History API: {start_date} → {end_date}")
        response = requests.get(url, params=params, verify=False, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        records = data.get('data', [])
        
        if records:
            df = pd.DataFrame(records)
            # Chỉ giữ lại các cột cần thiết cho weather
            weather_cols = ['datetime', 'temp', 'app_temp', 'rh', 'wind_spd', 'wind_dir', 
                           'pres', 'vis', 'clouds', 'precip', 'uv', 'dewpt']
            existing_cols = [col for col in weather_cols if col in df.columns]
            df = df[existing_cols]
            
            print(f"      Lấy được {len(df)} records weather data")
            return df
        else:
            print(f"      Không có weather data cho khoảng {start_date} → {end_date}")
            return None
            
    except Exception as e:
        print(f"      Lỗi weather API: {e}")
        return None


def get_air_quality_history(start_date, end_date, api_key):
    """
    Lấy dữ liệu chất lượng không khí lịch sử từ Weatherbit API.
    
    Parameters:
    -----------
    start_date : str
        Ngày bắt đầu (định dạng: 'YYYY-MM-DD')
    end_date : str
        Ngày kết thúc (định dạng: 'YYYY-MM-DD')
    api_key : str
        API key của Weatherbit
        
    Returns:
    --------
    df : pandas DataFrame hoặc None
        DataFrame chứa dữ liệu air quality hoặc None nếu có lỗi
    """
    url = "https://api.weatherbit.io/v2.0/history/airquality"
    params = {
        'lat': HANOI_LAT,
        'lon': HANOI_LON,
        'start_date': start_date,
        'end_date': end_date,
        'tz': 'local',
        'key': api_key
    }

    try:
        print(f"    Đang gọi Air Quality History API: {start_date} → {end_date}")
        response = requests.get(url, params=params, verify=False, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        records = data.get('data', [])
        
        if records:
            df = pd.DataFrame(records)
            # Chỉ giữ lại các cột cần thiết cho air quality
            air_cols = ['datetime', 'aqi', 'pm25', 'pm10', 'o3', 'so2', 'no2', 'co']
            existing_cols = [col for col in air_cols if col in df.columns]
            df = df[existing_cols]
            
            print(f"      Lấy được {len(df)} records air quality data")
            return df
        else:
            print(f"      Không có air quality data cho khoảng {start_date} → {end_date}")
            return None
            
    except Exception as e:
        print(f"      Lỗi air quality API: {e}")
        return None

In [3]:
def save_data_to_csv(data, file_name):
    """
    Lưu dữ liệu vào file CSV.
    
    Parameters:
    -----------
    data : pandas DataFrame
        DataFrame chứa dữ liệu cần lưu
    file_name : str
        Tên file CSV
    """
    if data is not None and not data.empty:
        # Kiểm tra file đã tồn tại chưa
        file_exists = os.path.isfile(file_name)
        
        # Lưu dữ liệu (append nếu file đã tồn tại)
        data.to_csv(
            file_name, 
            index=False, 
            mode='a' if file_exists else 'w',
            header=not file_exists
        )
        
        print(f"     Đã lưu {len(data)} bản ghi vào {file_name}")
        
        # Hiển thị tổng số bản ghi trong file
        if file_exists:
            total_records = len(pd.read_csv(file_name))
            print(f"     Tổng số bản ghi trong file: {total_records}")
    else:
        print("     Không có dữ liệu hợp lệ để lưu")


def process_date_range_separate(start_date, end_date, api_key):
    """
    Xử lý một khoảng thời gian: lấy weather data và air quality data riêng biệt.
    
    Parameters:
    -----------
    start_date : str
        Ngày bắt đầu
    end_date : str  
        Ngày kết thúc
    api_key : str
        API key
        
    Returns:
    --------
    tuple : (weather_df, air_quality_df)
        Tuple chứa 2 DataFrame hoặc None nếu có lỗi
    """
    print(f"\n Xử lý khoảng thời gian: {start_date} → {end_date}")
    
    # Lấy weather data
    weather_df = get_weather_data(start_date, end_date, api_key)
    
    # Lấy air quality history data 
    air_quality_df = get_air_quality_history(start_date, end_date, api_key)
    
    return weather_df, air_quality_df


def merge_and_aggregate_final_data():
    """
    Đọc 2 file CSV, sort theo thời gian, merge theo DATETIME (bao gồm giờ).
    
    Returns:
    --------
    merged_df : pandas DataFrame
        DataFrame đã được merge theo giờ (không aggregate)
    """
    print(f"\n Bắt đầu merge dữ liệu từ 2 file...")
    
    # Đọc file weather
    if os.path.isfile(WEATHER_FILE_TEST):
        print(f" Đọc file weather: {WEATHER_FILE_TEST}")
        weather_df = pd.read_csv(WEATHER_FILE_TEST)
        # Sửa format datetime: thay ':' cuối cùng bằng space
        weather_df['datetime'] = weather_df['datetime'].str.replace(r':(\d+)$', r' \1', regex=True)
        weather_df['datetime_parsed'] = pd.to_datetime(weather_df['datetime'])
        weather_df = weather_df.sort_values('datetime_parsed').reset_index(drop=True)
        print(f"    Weather data: {len(weather_df)} records")
    else:
        print(f" Không tìm thấy file weather: {WEATHER_FILE_TEST}")
        return None
    
    # Đọc file air quality
    if os.path.isfile(AIR_QUALITY_FILE_TEST):
        print(f"Đọc file air quality: {AIR_QUALITY_FILE_TEST}")
        air_quality_df = pd.read_csv(AIR_QUALITY_FILE_TEST)
        # Sửa format datetime: thay ':' cuối cùng bằng space
        air_quality_df['datetime'] = air_quality_df['datetime'].str.replace(r':(\d+)$', r' \1', regex=True)
        air_quality_df['datetime_parsed'] = pd.to_datetime(air_quality_df['datetime'])
        air_quality_df = air_quality_df.sort_values('datetime_parsed').reset_index(drop=True)
        print(f"    Air quality data: {len(air_quality_df)} records")
    else:
        print(f" Không tìm thấy file air quality: {AIR_QUALITY_FILE_TEST}")
        return None
    
    try:
        # Merge dữ liệu theo datetime ĐẦY ĐỦ (outer join để giữ tất cả dữ liệu)
        print(f" Merge dữ liệu theo datetime (bao gồm giờ)...")
        merged_df = pd.merge(weather_df, air_quality_df, on='datetime', how='outer', suffixes=('', '_aq'))
        
        # Lấy datetime từ cột không null
        merged_df['datetime_final'] = merged_df['datetime'].fillna(merged_df.get('datetime_aq', merged_df['datetime']))
        merged_df['datetime_parsed'] = pd.to_datetime(merged_df['datetime_final'])
        
        # Sort theo thời gian
        merged_df = merged_df.sort_values('datetime_parsed').reset_index(drop=True)
        print(f"    Merged hourly data: {len(merged_df)} records")
        
        # Tạo time features (GIỜ CŨNG ĐƯỢC GIỮ LẠI)
        print(f" Tạo time features...")
        merged_df['year'] = merged_df['datetime_parsed'].dt.year
        merged_df['month'] = merged_df['datetime_parsed'].dt.month
        merged_df['day'] = merged_df['datetime_parsed'].dt.day
        merged_df['hour'] = merged_df['datetime_parsed'].dt.hour
        merged_df['weekday'] = merged_df['datetime_parsed'].dt.weekday
        
        # Chọn các cột cần thiết
        # Weather columns
        weather_cols = ['temp', 'app_temp', 'rh', 'wind_spd', 'wind_dir', 'pres', 'vis', 'clouds', 'precip', 'uv', 'dewpt']
        existing_weather = [col for col in weather_cols if col in merged_df.columns]
        
        # Air quality columns
        air_cols = ['aqi', 'pm25', 'pm10', 'o3', 'so2', 'no2', 'co']
        existing_air = [col for col in air_cols if col in merged_df.columns]
        
        # Reorder columns: time features + weather + air quality
        base_cols = ['datetime_parsed', 'year', 'month', 'day', 'hour', 'weekday']
        ordered_cols = base_cols + existing_weather + existing_air
        
        # Chỉ giữ các cột tồn tại
        final_cols = [col for col in ordered_cols if col in merged_df.columns]
        final_df = merged_df[final_cols].copy()
        
        # Đổi tên cột datetime_parsed thành datetime
        final_df.rename(columns={'datetime_parsed': 'datetime'}, inplace=True)
        
        print(f"    Final hourly data: {len(final_df)} records")
        print(f"    Columns: {final_df.columns.tolist()}")
        
        return final_df
        
    except Exception as e:
        print(f" Lỗi khi merge data: {e}")
        import traceback
        traceback.print_exc()
        return None

In [4]:
def get_months_between_dates(start_date, end_date):
    """
    Lấy danh sách các khoảng thời gian theo tháng giữa hai ngày.
    
    Parameters:
    -----------
    start_date : datetime
        Ngày bắt đầu
    end_date : datetime
        Ngày kết thúc
        
    Returns:
    --------
    months : list of tuples
        Danh sách các tuple (start_date, end_date) cho mỗi tháng
    """
    months = []
    start_year, start_month = start_date.year, start_date.month
    end_year, end_month = end_date.year, end_date.month

    while start_year < end_year or (start_year == end_year and start_month <= end_month):
        # Tạo ngày bắt đầu tháng
        month_start = datetime(start_year, start_month, 1)
        
        # Lấy ngày cuối tháng
        _, last_day = monthrange(start_year, start_month)
        month_end = datetime(start_year, start_month, last_day)
        
        # Đảm bảo không vượt quá end_date
        if month_end > end_date:
            month_end = end_date

        months.append((
            month_start.strftime('%Y-%m-%d'), 
            month_end.strftime('%Y-%m-%d')
        ))

        # Chuyển sang tháng tiếp theo
        if start_month == 12:
            start_month = 1
            start_year += 1
        else:
            start_month += 1

    return months

In [5]:
print("="*60)
print("CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT + CHẤT LƯỢNG KHÔNG KHÍ")
print("="*60)
print(f"Tọa độ Hà Nội: {HANOI_LAT}, {HANOI_LON}")
print(f"Từ ngày: {start_date.strftime('%Y-%m-%d')}")
print(f"Đến ngày: {end_date.strftime('%Y-%m-%d')}")
print(f"\nFile outputs:")
print(f"   Weather data: {WEATHER_FILE_TEST}")
print(f"   Air quality data: {AIR_QUALITY_FILE_TEST}")
print(f"   Final combined: {FINAL_COMBINED_FILE}")

# Lấy danh sách các tháng
months = get_months_between_dates(start_date, end_date)
print(f"\nTổng số tháng cần lấy dữ liệu: {len(months)}")
print(f" API 1: Weather hourly history (https://api.weatherbit.io/v2.0/history/hourly)")
print(f" API 2: Air quality history (https://api.weatherbit.io/v2.0/history/airquality)")
print(f" Strategy: Lưu riêng từng API → Sort theo thời gian → Merge → Daily aggregate")
print(f"\nDanh sách các khoảng thời gian:")
for i, (month_start, month_end) in enumerate(months[:5], 1):
    print(f"   {i}. {month_start} → {month_end}")
if len(months) > 5:
    print(f"   ... và {len(months) - 5} tháng khác")
print("="*60)

CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT + CHẤT LƯỢNG KHÔNG KHÍ
Tọa độ Hà Nội: 21.0285, 105.8542
Từ ngày: 2025-11-01
Đến ngày: 2025-11-07

File outputs:
   Weather data: hanoi_weather_history_test_data.csv
   Air quality data: hanoi_air_quality_history_test_data.csv
   Final combined: hanoi_weather_air_quality_final_test_data.csv

Tổng số tháng cần lấy dữ liệu: 1
 API 1: Weather hourly history (https://api.weatherbit.io/v2.0/history/hourly)
 API 2: Air quality history (https://api.weatherbit.io/v2.0/history/airquality)
 Strategy: Lưu riêng từng API → Sort theo thời gian → Merge → Daily aggregate

Danh sách các khoảng thời gian:
   1. 2025-11-01 → 2025-11-07


In [6]:
print("\n" + "="*60)
print("CHUẨN BỊ LẤY DỮ LIỆU")
print("="*60 + "\n")

# Xóa các file cũ nếu có để bắt đầu fresh
for file_path in [WEATHER_FILE_TEST, AIR_QUALITY_FILE_TEST]:
    if os.path.isfile(file_path):
        os.remove(file_path)
        print(f" Đã xóa file cũ: {file_path}")

print(f"\n Sẵn sàng lấy dữ liệu từ {len(months)} tháng")
print(f" Weather file: {WEATHER_FILE_TEST}")
print(f" Air Quality file: {AIR_QUALITY_FILE_TEST}")
print(f" Final file: {FINAL_COMBINED_FILE}")
print("="*60)


CHUẨN BỊ LẤY DỮ LIỆU


 Sẵn sàng lấy dữ liệu từ 1 tháng
 Weather file: hanoi_weather_history_test_data.csv
 Air Quality file: hanoi_air_quality_history_test_data.csv
 Final file: hanoi_weather_air_quality_final_test_data.csv


In [8]:
# BƯỚC 1A: LẤY DỮ LIỆU WEATHER (API 1)
print("\n" + "="*60)
print("BƯỚC 1A: LẤY DỮ LIỆU WEATHER API")
print("="*60 + "\n")

# Thống kê weather
weather_success = 0
weather_failed = 0
total_months = len(months)

# Lấy dữ liệu weather cho từng tháng
for index, (month_start, month_end) in enumerate(months, 1):
    print(f"\n[{index}/{total_months}]  Weather API: {month_start} → {month_end}")
    
    # Lấy weather data
    weather_df = get_weather_data(month_start, month_end, API_KEY)
    
    # Lưu weather data
    if weather_df is not None and not weather_df.empty:
        save_data_to_csv(weather_df, WEATHER_FILE_TEST)
        weather_success += 1
        print(f"     Weather: {len(weather_df)} records saved")
    else:
        weather_failed += 1
        print(f"     Weather: No data")
    
    # Delay để tránh rate limiting
    if index < total_months:
        print(f"     Waiting 2 seconds...")
        time.sleep(2)

# Tóm tắt kết quả Weather API
print("\n" + "="*50)
print("KẾT QUẢ WEATHER API")
print("="*50)
print(f" Thành công: {weather_success}/{total_months} tháng")
print(f" Thất bại: {weather_failed}/{total_months} tháng")

# Hiển thị thông tin file weather
if os.path.isfile(WEATHER_FILE_TEST):
    weather_df_final = pd.read_csv(WEATHER_FILE_TEST)
    print(f" Weather file: {len(weather_df_final):,} records")
    print(f" Time range: {weather_df_final['datetime'].min()} → {weather_df_final['datetime'].max()}")
else:
    print(f" Weather file không tồn tại")

print("="*50)


BƯỚC 1A: LẤY DỮ LIỆU WEATHER API


[1/1]  Weather API: 2025-11-01 → 2025-11-07
    Đang gọi Weather History API: 2025-11-01 → 2025-11-07
      Lỗi weather API: 429 Client Error: Too Many Requests for url: https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-11-01&end_date=2025-11-07&tz=local&key=6fd51f49cdc045cf862c975a9f53380e
     Weather: No data

KẾT QUẢ WEATHER API
 Thành công: 0/1 tháng
 Thất bại: 1/1 tháng
 Weather file không tồn tại


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# BƯỚC 1B: LẤY DỮ LIỆU AIR QUALITY (API 2) 
print("\n" + "="*60)
print("BƯỚC 1B: LẤY DỮ LIỆU AIR QUALITY API")
print("="*60 + "\n")

# Thống kê air quality
air_quality_success = 0
air_quality_failed = 0

# # Lấy dữ liệu air quality cho từng tháng
for index, (month_start, month_end) in enumerate(months, 1):
    print(f"\n[{index}/{total_months}]  Air Quality API: {month_start} → {month_end}")
    
    # Lấy air quality data
    air_quality_df = get_air_quality_history(month_start, month_end, API_KEY_BACKUP)
    
    # Lưu air quality data
    if air_quality_df is not None and not air_quality_df.empty:
        save_data_to_csv(air_quality_df, AIR_QUALITY_FILE_TEST)
        air_quality_success += 1
        print(f"     Air Quality: {len(air_quality_df)} records saved")
    else:
        air_quality_failed += 1
        print(f"     Air Quality: No data")
    
    # Delay để tránh rate limiting
    if index < total_months:
        print(f"     Waiting 2 seconds...")
        time.sleep(2)

# Tóm tắt kết quả Air Quality API
print("\n" + "="*50)
print("KẾT QUẢ AIR QUALITY API")
print("="*50)
print(f" Thành công: {air_quality_success}/{total_months} tháng")
print(f" Thất bại: {air_quality_failed}/{total_months} tháng")

# Hiển thị thông tin file air quality
if os.path.isfile(AIR_QUALITY_FILE_TEST):
    air_quality_df_final = pd.read_csv(AIR_QUALITY_FILE_TEST)
    print(f" Air Quality file: {len(air_quality_df_final):,} records")
    print(f" Time range: {air_quality_df_final['datetime'].min()} → {air_quality_df_final['datetime'].max()}")
else:
    print(f" Air Quality file không tồn tại")

print("="*50)

In [ ]:
# BƯỚC 2: MERGE DỮ LIỆU TỪ 2 FILE
print("\n" + "="*60)
print("BƯỚC 2: MERGE VÀ TẠO FILE CUỐI CÙNG")
print("="*60)

# Merge và tạo file cuối cùng
final_df = merge_and_aggregate_final_data()

if final_df is not None and not final_df.empty:
    # Lưu file cuối cùng
    final_df.to_csv(FINAL_COMBINED_FILE, index=False)
    print(f"\n Đã lưu file cuối cùng: {FINAL_COMBINED_FILE}")
    print(f" Tổng số records: {len(final_df):,}")
    
    # Đọc và hiển thị thông tin file CSV cuối cùng
    print(f"\n THÔNG TIN FILE CSV CUỐI CÙNG:")
    print("-" * 60)
    
    print(f"    Tổng số dòng: {len(final_df):,}")
    print(f"    Tổng số cột: {len(final_df.columns)}")
    
    if len(final_df) > 0:
        print(f"    Khoảng thời gian: {final_df['datetime'].min()} → {final_df['datetime'].max()}")
        
        print(f"\n    Các cột trong dataset:")
        for i, col in enumerate(final_df.columns, 1):
            print(f"      {i:2d}. {col}")
        
        # Kiểm tra các cột quan trọng
        important_cols = []
        for col in final_df.columns:
            if 'pm25' in col.lower():
                important_cols.append(col)
            elif 'temp' in col.lower() and 'mean' in col.lower():
                important_cols.append(col)
            elif 'aqi' in col.lower():
                important_cols.append(col)
        
        if important_cols:
            print(f"\n    Thống kê cơ bản:")
            for col in important_cols[:3]:  # Chỉ hiển thị 3 cột đầu
                if col in final_df.columns:
                    print(f"      {col}: {final_df[col].min():.1f} - {final_df[col].max():.1f} (mean: {final_df[col].mean():.1f})")
        
        print(f"\n    Preview 5 dòng đầu:")
        display_cols = ['datetime', 'year', 'month', 'day']
        
        # Thêm một số cột quan trọng
        for col in final_df.columns:
            if len(display_cols) < 8:
                if 'temp' in col.lower() and 'mean' in col.lower():
                    display_cols.append(col)
                elif 'pm25' in col.lower():
                    display_cols.append(col)
                elif 'aqi' in col.lower():
                    display_cols.append(col)
        
        existing_display_cols = [col for col in display_cols if col in final_df.columns]
        if existing_display_cols:
            print(final_df[existing_display_cols].head().to_string(index=False))
        
        print(f"\n    Dataset sẵn sàng cho model training PM2.5!")
        
        # Thông tin missing values
        print(f"\n    Missing values:")
        missing_info = final_df.isnull().sum()
        missing_percent = (missing_info / len(final_df) * 100).round(1)
        for col in missing_info.index:
            if missing_info[col] > 0:
                print(f"      {col}: {missing_info[col]} ({missing_percent[col]}%)")
    else:
        print(f"    File CSV rỗng")
    
else:
    print(f"\n Không thể tạo file cuối cùng")

print("\n" + "="*60)
print("HOÀN THÀNH")
print("="*60)